# Check the export

A quick look at what `run_exporting_pipeline.py` wrote to `<neuropixels_dir>/<export_folder_name>/export/`.
Set the two file paths in the first cell; nothing else is needed -- no session config, no
`spikesorting` package, just h5py:

1. **`<recording>.sorted_spikes.mat`** -- how many units, how many single units (`good`) and
   multi-units (`mua`), where they sit on the probe (each unit's mean waveform at Kilosort's own
   spike position, as `export/figures/overview.pdf` draws it), their firing rates, and a few
   random examples;
2. **`<run>.lfp.mat`** (when the session has `export_lfp: true`) -- a few LFP traces and their
   spectrogram.

Plain h5py: a v7.3 `.mat` is an HDF5 file. Two things to know when reading one in Python --
MATLAB stores every array **transposed** (h5py's shape is MATLAB's reversed), and text as
**character codes** (`"".join(map(chr, ...))` turns it back into a string).

In [ ]:
import json
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal

# ---------------- edit these: the two exported files ----------------
# Both sit in <neuropixels_dir>/kilosort4/export/ (or wherever that folder was copied).
SPIKES_MAT = Path(r"Z:/Monkey Athos/2026-11-10/.../kilosort4/export/<recording>.sorted_spikes.mat")
LFP_MAT = Path(r"Z:/Monkey Athos/2026-11-10/.../kilosort4/export/<run>.lfp.mat")   # or None: no LFP section

for path in (SPIKES_MAT, LFP_MAT):
    if path is not None:
        print(f"{path.name:60} {path.stat().st_size / 1e6:10.1f} MB")

## 1. Load the sorted spikes

`sorted_spikes` has one entry per **spike** (`TimeStamps` in s, `Unit`, `Channel`) and an `info`
struct with one entry per **unit** (`Unit_No`, `Label`, `KSLabel`, `Channel_Number`, `n_spikes`).
`MeanWaveform`/`StdWaveform` hold each unit's measured mean ± std on its peak channel. Units Phy
labelled `noise` are not exported.

In [ ]:
# Every file is opened in a `with` block and closed when the block ends. An open
# h5py file stays locked for as long as this notebook's kernel runs, and the
# exporting pipeline then cannot rewrite it ("unable to lock file").
with h5py.File(SPIKES_MAT, "r") as f:
    s = f["sorted_spikes"]
    print(SPIKES_MAT.name)
    for name in list(s) + [f"info/{k}" for k in s["info"]]:
        if isinstance(s[name], h5py.Dataset):
            print(f"  {name:28} MATLAB size {' x '.join(map(str, s[name].shape[::-1]))}")

In [ ]:
with h5py.File(SPIKES_MAT, "r") as f:                # read into memory, then closed
    s = f["sorted_spikes"]
    times = s["TimeStamps"][()].ravel()                  # every spike, seconds
    spike_unit = s["Unit"][()].ravel().astype(int)

    unit_ids = s["info/Unit_No"][()].ravel().astype(int)
    labels = "".join(map(chr, s["info/Label"][()].ravel())).split(" | ")
    ks_labels = "".join(map(chr, s["info/KSLabel"][()].ravel())).split(" | ")
    channels = s["info/Channel_Number"][()].ravel()
    n_spikes = s["info/n_spikes"][()].ravel().astype(int)
    timebase = "".join(map(chr, s["info/timebase"][()].ravel()))
    fs = s["TimeRes"][()].item()

    has_waves = "MeanWaveform" in s                  # each unit's measured mean, for section 5
    if has_waves:
        mean_wave = s["MeanWaveform"][()]              # (samples, units): MATLAB's nUnits x nSamp
        std_wave = s["StdWaveform"][()]
        wave_ids = list(s["info/MeanWaveform_Unit_No"][()].ravel().astype(int))
        wave_unit = "".join(map(chr, s["MeanWaveformUnit"][()].ravel()))

# Firing rates divide by the recording length the export used, when the manifest beside it says it.
manifest = SPIKES_MAT.parent / "sorting_summary_info.json"
window = json.loads(manifest.read_text()).get("recording_window", {}) if manifest.exists() else {}
duration_s = window.get("duration_s", times.max() - times.min())

print(f"{times.size:,} spikes, {unit_ids.size} units, timebase {timebase}")
print(f"recording: {duration_s / 60:.1f} min ({window.get('from', 'span of the spikes')})")

## 2. How many units, and what kind

`good` = single unit, `mua` = multi-unit. A unit nobody curated in Phy keeps Kilosort's own label.

In [ ]:
units = pd.DataFrame({
    "unit_id": unit_ids,
    "label": labels,
    "ks_label": ks_labels,
    "channel": channels,
    "n_spikes": n_spikes,
    "rate_hz": n_spikes / duration_s,
})
units["kind"] = units["label"].map({"good": "single unit", "mua": "multi-unit"}).fillna(units["label"])

print(units["kind"].value_counts().to_string(), "\n")
print(pd.crosstab(units["label"], units["ks_label"], rownames=["label (curated)"], colnames=["KSLabel"]))

counts = units["kind"].value_counts()
plt.figure(figsize=(5, 3.5))
bars = plt.bar(counts.index, counts.values, color=[f"C{i}" for i in range(len(counts))])
plt.bar_label(bars)
plt.ylabel("units")
plt.title(f"{len(units)} units exported")
plt.show()

## 3. The probe overview: every unit where it sits

The same pictures as `export/figures/overview.pdf`, built here from the files:

- **spike positions** are Kilosort's own (`spike_positions.npy`, one x/y per spike), in the
  sorting folder that holds `export/`; a unit sits at the median of its spikes;
- each unit's **mean waveform** is the measured one from `sorted_spikes.mat` (section 1), drawn
  at its place on the probe, all on one height scale so a bigger unit looks bigger;
- **banks** come from the recording's `.meta` (its `~imroTbl`, NP 1.0-format probes only),
  found through `export/run_info.json`. Without it there is no bank bar.

Depth is distance from the probe tip.

In [ ]:
SORT_DIR = SPIKES_MAT.parent.parent          # the sorting folder that holds export/
MAX_SPIKES = 100_000                         # evenly spread, as the export's overview

positions_all = np.load(SORT_DIR / "spike_positions.npy", mmap_mode="r")      # Kilosort's x/y per spike
clusters_all = np.load(SORT_DIR / "spike_clusters.npy", mmap_mode="r").ravel()  # after Phy curation
sites = np.load(SORT_DIR / "channel_positions.npy")                          # one x/y per site
chan_map = np.load(SORT_DIR / "channel_map.npy").ravel()                     # each site's binary row

pick = np.unique(np.linspace(0, clusters_all.size - 1, min(MAX_SPIKES, clusters_all.size)).round().astype(int))
xy = np.asarray(positions_all[pick], dtype=float)
cl = np.asarray(clusters_all[pick])
unit_xy = {u: np.median(xy[cl == u], axis=0) for u in unit_ids if np.any(cl == u)}
print(f"{pick.size:,} of {clusters_all.size:,} spikes; {len(unit_xy)} of {unit_ids.size} units placed")
print(f"sites: x {sorted(set(sites[:, 0].tolist()))[:6]}, y {sites[:, 1].min():g}-{sites[:, 1].max():g} um")

# Banks, from the .meta beside the recording. NP 1.0-format imro entries are
# (chan bank refid apgain lfgain ...); other formats put the bank elsewhere, so no bar.
site_bank = None
run_info = SPIKES_MAT.parent / "run_info.json"
meta_path = (Path(json.loads(run_info.read_text())["source_recording"]).with_suffix(".meta")
             if run_info.exists() else None)
if meta_path is not None and meta_path.exists():
    meta = dict(l.split("=", 1) for l in meta_path.read_text().splitlines() if "=" in l)
    np1_types = {0, 1020, 1030, 1100, 1120, 1121, 1122, 1123, 1200, 1300}
    entries = [e.strip("( ") for e in meta.get("~imroTbl", "").split(")") if e.strip("( ")][1:]
    if int(meta.get("imDatPrb_type", "0")) in np1_types and entries:
        bank_of_chan = {int(e.split()[0]): int(e.split()[1]) for e in entries}
        site_bank = np.array([bank_of_chan.get(int(c), -1) for c in chan_map])
        print("banks recorded:", sorted(set(site_bank.tolist()) - {-1}))
print("bank bar:", "from " + meta_path.name if site_bank is not None else "none (no readable NP 1.0 .meta)")

In [ ]:
# Every exported unit's sampled spikes on the probe (the PDF's page 1); others in grey.
exported = np.isin(cl, unit_ids)
plt.figure(figsize=(4, 8))
plt.scatter(xy[~exported, 0], xy[~exported, 1], s=1, color="0.85", linewidths=0)
plt.scatter(xy[exported, 0], xy[exported, 1], s=1, c=cl[exported] % 20, cmap="tab20", linewidths=0)
plt.xlabel("x (um)")
plt.ylabel("depth from tip (um)")
plt.title(f"spike positions, {np.unique(cl[exported]).size} units")
plt.show()

In [ ]:
# Each unit's measured mean waveform at its place on the probe: one colour per unit,
# then single units red and multi-units light blue (drawn last, on top).
GLYPH_WIDTH_UM, GLYPH_HEIGHT_UM = 30.0, 40.0      # glyph width; height of the median unit
BANK_COLOURS = ["#2e7d32", "#66bb6a", "#c62828", "#d81b60", "#1e88e5", "#8e24aa"]

drawn = [u for u in wave_ids if u in unit_xy] if has_waves else []
ptp = np.ptp(mean_wave, axis=0) if has_waves else np.ones(1)
scale = GLYPH_HEIGHT_UM / np.median(ptp[ptp > 0])           # um per waveform unit, one for all
offsets = np.linspace(-GLYPH_WIDTH_UM / 2, GLYPH_WIDTH_UM / 2, mean_wave.shape[0]) if has_waves else []
kind_of = dict(zip(units["unit_id"], units["kind"]))
kind_colour = {"single unit": "#d32f2f", "multi-unit": "#64b5f6"}

for colour_by in ("unit", "kind"):
    fig, (ax_bank, ax) = plt.subplots(1, 2, figsize=(4.4, 11), sharey=True,
                                      gridspec_kw={"width_ratios": [0.22, 1], "wspace": 0.02})
    ax.scatter(sites[:, 0], sites[:, 1], s=2, color="0.85", linewidths=0)
    order = drawn if colour_by == "unit" else sorted(drawn, key=lambda u: kind_of.get(u) == "single unit")
    for k, u in enumerate(order):
        x0, y0 = unit_xy[u]
        w = mean_wave[:, wave_ids.index(u)]
        colour = plt.cm.tab20(k % 20) if colour_by == "unit" else kind_colour.get(kind_of.get(u), "0.6")
        ax.plot(x0 + offsets, y0 + scale * w, color=colour, lw=0.8, alpha=0.9)
    if colour_by == "kind":
        for name, colour in kind_colour.items():
            ax.plot([], [], color=colour, label=f"{name} ({sum(kind_of.get(u) == name for u in drawn)})")
        ax.legend(loc="lower right", bbox_to_anchor=(1.0, 1.0), fontsize=7, frameon=False)
    x_lo, y_lo = sites[:, 0].min() - 40, sites[:, 1].min() - 60
    ax.set_xlim(x_lo, sites[:, 0].max() + 40)
    ax.set_ylim(y_lo, sites[:, 1].max() + 60)
    ax.axis("off")
    ax.plot([x_lo + 5, x_lo + 130], [y_lo + 15] * 2, color="k", lw=1.2)        # 125 um scale bar
    ax.text(x_lo + 67, y_lo + 5, "125 um", ha="center", va="top", fontsize=8)
    ax.set_title(f"{unit_ids.size:,} units", fontsize=9, loc="left")
    if site_bank is not None:                    # one bar segment per run of sites in a bank
        order_y = np.argsort(sites[:, 1])
        ys, banks = sites[order_y, 1], site_bank[order_y]
        half = np.min(np.diff(np.unique(ys))) / 2
        start = 0
        for i in range(1, ys.size + 1):
            if i == ys.size or banks[i] != banks[start]:
                if banks[start] >= 0:
                    lo, hi = ys[start] - half, ys[i - 1] + half
                    ax_bank.add_patch(plt.Rectangle((0.55, lo), 0.3, hi - lo, lw=0,
                                                    color=BANK_COLOURS[banks[start] % len(BANK_COLOURS)]))
                    ax_bank.text(0.35, (lo + hi) / 2, f"Bank {banks[start]}", rotation=90,
                                 ha="right", va="center", fontsize=8)
                start = i
    ax_bank.set_xlim(0, 1)
    ax_bank.set_xticks([])
    for side in ("top", "right", "bottom"):
        ax_bank.spines[side].set_visible(False)
    ax_bank.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1000:g}"))
    ax_bank.set_ylabel("distance from probe tip (mm)")
    plt.show()

## 4. Firing-rate distribution

In [ ]:
print(units.groupby("kind")["rate_hz"].describe().round(2), "\n")

edges = np.logspace(np.log10(units["rate_hz"].min() * 0.9), np.log10(units["rate_hz"].max() * 1.1), 30)
plt.figure(figsize=(9, 4))
for k in counts.index:
    plt.hist(units.loc[units["kind"] == k, "rate_hz"], bins=edges, histtype="step", linewidth=2, label=k)
plt.xscale("log")
plt.xlabel("firing rate (Hz)")
plt.ylabel("units")
plt.legend()
plt.show()

## 5. A few random examples

`N_EXAMPLES` units of each kind: mean waveform ± std, ISI histogram (dashed line at 1.5 ms -- a
single unit has almost nothing left of it), and firing rate over time in 60 s bins.

In [ ]:
N_EXAMPLES = 4
SEED = 0              # change it to see other units

if has_waves:                                         # read in section 1
    t_ms = (np.arange(mean_wave.shape[0]) - mean_wave.shape[0] // 2) / fs * 1000

rng = np.random.default_rng(SEED)
for kind in counts.index:
    ids = units.loc[units["kind"] == kind, "unit_id"].to_numpy()
    ids = rng.choice(ids, size=min(N_EXAMPLES, ids.size), replace=False)
    fig, axes = plt.subplots(len(ids), 3, figsize=(14, 2.4 * len(ids)), squeeze=False,
                             gridspec_kw={"width_ratios": [1, 1.2, 2]})
    for (ax_w, ax_i, ax_r), u in zip(axes, ids):
        t = np.sort(times[spike_unit == u])
        row = units.set_index("unit_id").loc[u]

        if has_waves and u in wave_ids:
            w, sd = mean_wave[:, wave_ids.index(u)], std_wave[:, wave_ids.index(u)]
            ax_w.plot(t_ms, w, color="k")
            ax_w.fill_between(t_ms, w - sd, w + sd, color="k", alpha=0.2)
            ax_w.set_ylabel(wave_unit)
        ax_w.set_xlabel("time (ms)")
        ax_w.set_title(f"unit {u}, ch {row['channel']:.0f}", fontsize=9, loc="left")

        isi_ms = np.diff(t) * 1000
        ax_i.hist(isi_ms[isi_ms < 50], bins=np.arange(0, 50.5, 0.5), color="0.3")
        ax_i.axvline(1.5, color="crimson", linestyle="--")
        ax_i.set_xlabel("ISI (ms)")
        ax_i.set_title(f"ISI < 1.5 ms: {np.mean(isi_ms < 1.5):.2%}", fontsize=9, loc="left")

        rate, bin_edges = np.histogram(t, bins=np.arange(t.min(), t.max() + 60, 60))
        ax_r.plot(bin_edges[:-1] + 30, rate / 60)
        ax_r.set_ylim(bottom=0)
        ax_r.set_xlabel("time (s)")
        ax_r.set_ylabel("rate (Hz)")
        ax_r.set_title(f"{row['n_spikes']:,} spikes, {row['rate_hz']:.2f} Hz", fontsize=9, loc="left")
    fig.suptitle(f"{kind}: {len(ids)} of {int(counts[kind])}")
    plt.tight_layout()
    plt.show()

## 6. LFP: a few traces and their spectrogram

`lfp.data` is int16, MATLAB size nChan x nSamples, at `fs` (2500 Hz / `lfp_decimate`).
Microvolts = `data * uv_per_digit` per channel; row `sy_index` is the SY word, not a channel.
Sample `k` is at `t0 + k / fs` seconds on the probe's clock.

The window is in **seconds from the start of the recording**. The spectrogram resolves
`1 / SPEC_SEGMENT_S` Hz, so the window should be several segments long.

In [ ]:
LFP_WINDOW_S = (60.0, 100.0)      # (start, stop), seconds
LFP_CHANNELS = [0, 100, 200, 300] # rows of the LF binary
SPEC_SEGMENT_S = 1.0              # 1 s segments -> 1 Hz resolution
FREQ_RANGE_HZ = (1, 150)

assert LFP_MAT is not None, "no LFP file named: set LFP_MAT at the top (export_lfp: true in the session writes one)"
with h5py.File(LFP_MAT, "r") as f:               # closed when the block ends
    lf = f["lfp"]
    print(LFP_MAT.name)
    for name in lf:
        value = lf[name][()] if lf[name].size <= 4 or lf[name].attrs["MATLAB_class"] == b"char" else None
        if lf[name].attrs["MATLAB_class"] == b"char":
            value = "".join(map(chr, value.ravel()))
        elif value is None:
            value = f"[{' x '.join(map(str, lf[name].shape[::-1]))}]"
        elif value.size == 1:
            value = value.item()
        print(f"  {name:20} {str(value)[:90]}")
    lfp_fs = lf["fs"][()].item()
    t0 = lf["t0"][()].item()
    uv_per_digit = lf["uv_per_digit"][()].ravel()

In [ ]:
k0, k1 = int((LFP_WINDOW_S[0] - t0) * lfp_fs), int((LFP_WINDOW_S[1] - t0) * lfp_fs)

with h5py.File(LFP_MAT, "r") as f:                  # only the window is read, then closed
    raw = f["lfp/data"][k0:k1, :]                   # (samples, channels)
lfp_uv = raw[:, LFP_CHANNELS] * uv_per_digit[LFP_CHANNELS]
lfp_t = t0 + np.arange(k0, k1) / lfp_fs
print(f"{lfp_uv.shape[0]:,} samples x {len(LFP_CHANNELS)} channels, {lfp_t[0]:.1f}-{lfp_t[-1]:.1f} s")

spacing = 1.2 * np.ptp(np.percentile(lfp_uv, [0.5, 99.5], axis=0), axis=0).max()
plt.figure(figsize=(14, 1.2 * len(LFP_CHANNELS) + 1.5))
for i, ch in enumerate(LFP_CHANNELS):
    plt.plot(lfp_t, lfp_uv[:, i] - np.median(lfp_uv[:, i]) + i * spacing, lw=0.5, color="0.15")
plt.yticks(np.arange(len(LFP_CHANNELS)) * spacing, [f"ch {c}" for c in LFP_CHANNELS])
plt.xlabel("time (s)")
plt.title(f"LFP, traces {spacing:.0f} uV apart")
plt.show()

In [ ]:
fig, axes = plt.subplots(len(LFP_CHANNELS), 2, figsize=(14, 2.4 * len(LFP_CHANNELS)), squeeze=False,
                         sharex="col", gridspec_kw={"width_ratios": [3, 1]}, constrained_layout=True)
nperseg = int(SPEC_SEGMENT_S * lfp_fs)
for (ax_s, ax_p), i, ch in zip(axes, range(len(LFP_CHANNELS)), LFP_CHANNELS):
    x = lfp_uv[:, i] - lfp_uv[:, i].mean()
    freq, seg_t, power = signal.spectrogram(x, fs=lfp_fs, nperseg=nperseg, noverlap=nperseg // 2)
    keep = (freq >= FREQ_RANGE_HZ[0]) & (freq <= FREQ_RANGE_HZ[1])
    mesh = ax_s.pcolormesh(lfp_t[0] + seg_t, freq[keep], 10 * np.log10(power[keep]), shading="nearest")
    fig.colorbar(mesh, ax=ax_s, label="dB")
    ax_s.set_ylabel(f"ch {ch}\nfrequency (Hz)")

    freq, psd = signal.welch(x, fs=lfp_fs, nperseg=nperseg)
    keep = (freq >= FREQ_RANGE_HZ[0]) & (freq <= FREQ_RANGE_HZ[1])
    ax_p.semilogy(freq[keep], psd[keep], color="0.2")
    ax_p.set_ylabel("PSD (uV^2/Hz)")
axes[-1, 0].set_xlabel("time (s)")
axes[-1, 1].set_xlabel("frequency (Hz)")
plt.show()